<div align="center">

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rajeevraibhatia/agent-harness-evals/blob/main/notebooks/m1_react_loop.ipynb)
[![Course](https://img.shields.io/badge/Full%20Course-rajeevraibhatia.com-7c3aed)](https://rajeevraibhatia.com/curriculum/agent-harness-evals#module-1)

</div>

# Module 1 — Agent Architecture Taxonomy

**Level:** Medium | **Time:** ~1h | [Full reading →](https://rajeevraibhatia.com/curriculum/agent-harness-evals#module-1)

### What you'll build
A ReAct (Reasoning + Acting) loop from scratch using the OpenAI SDK — no LangChain, no frameworks.

### Key concepts
- **Workflows vs agents**: workflows = predetermined LLM call sequences; agents = LLM directs its own control flow
- **6 building blocks** (Anthropic taxonomy): augmented LLM → prompt chaining → routing → parallelization → orchestrator-workers → evaluator-optimizer
- **ReAct pattern** (Yao 2022): `Thought → Action → Observation` cycles until final answer
- **ACI (Agent-Computer Interface)**: tool APIs deserve the same design rigor as HCI

### Research refs
- ReAct: Synergizing Reasoning and Acting — Yao et al. (2022) https://arxiv.org/abs/2210.03629
- Tree of Thoughts — Yao et al. (2023) https://arxiv.org/abs/2305.10601

---

In [ ]:
# Install dependencies
!pip install openai --quiet

In [ ]:
import os
from openai import OpenAI

client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])

TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "search",
            "description": "Search the web and return relevant text snippets. Use for current facts, recent events, and verifiable information. Be precise — 'Paris metro area population 2024' beats 'Paris population'.",
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {
                        "type": "string",
                        "description": "Specific search query. Max 200 chars."
                    }
                },
                "required": ["query"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "calculator",
            "description": "Evaluate a mathematical expression. Examples: '20.1 / 12.3', 'sqrt(144)'.",
            "parameters": {
                "type": "object",
                "properties": {
                    "expression": {"type": "string"}
                },
                "required": ["expression"]
            }
        }
    }
]

def mock_tool_executor(name, args):
    """Mock tool executor — swap in real implementations."""
    if name == "search":
        q = args.get("query", "").lower()
        if "paris" in q:
            return "Ile-de-France (Greater Paris) had ~12.3 million residents in 2024."
        if "new york" in q or "nyc" in q:
            return "Greater NYC metro area had ~20.1 million residents in 2024."
        return "No results found."
    if name == "calculator":
        try:
            import math
            result = eval(args["expression"], {"__builtins__": {}}, vars(math))
            return str(result)
        except Exception as e:
            return f"Error: {e}"
    return f"Unknown tool: {name}"

def react_loop(question: str, max_steps: int = 10) -> str:
    messages = [
        {"role": "system", "content": "You are a helpful assistant. Think step by step. Use tools when you need external data or calculations. Reason explicitly before each tool call."},
        {"role": "user", "content": question}
    ]

    for step in range(max_steps):
        response = client.chat.completions.create(
            model="gpt-4o",
            messages=messages,
            tools=TOOLS,
            tool_choice="auto"
        )
        msg = response.choices[0].message
        messages.append(msg)

        if not msg.tool_calls:
            # No tool calls → final answer
            return msg.content

        # Execute all requested tool calls
        for tc in msg.tool_calls:
            name = tc.function.name
            args = json.loads(tc.function.arguments)
            print(f"  [step {step+1}] {name}({args})")
            observation = mock_tool_executor(name, args)
            print(f"           → {observation}")
            messages.append({
                "role": "tool",
                "tool_call_id": tc.id,
                "content": observation
            })

    return "Max steps reached without final answer."

import json

question = "How much larger is the New York metro area than Paris? Express as a percentage."
print(f"Question: {question}\n")
answer = react_loop(question)
print(f"\nAnswer: {answer}")

## Exercise

For each task below, choose the right architecture and justify the trade-offs.

> **Interview question:** *"What's the difference between a workflow and an agent? Give an example of each for a customer support product."*

In [ ]:
# Exercise: Architecture Selection
# For each task below, choose: (a) single LLM call, (b) workflow, or (c) agent loop
# Justify latency and cost trade-offs for each.

tasks = [
    "Extract name, date, and amount from a PDF invoice.",
    "Write a 5-page research report on climate policy, citing 10+ sources.",
    "Classify customer support tickets into 5 buckets (billing, technical, account, feature, other)."
]

# Your analysis:
for i, task in enumerate(tasks, 1):
    print(f"Task {i}: {task}")
    # TODO: add your architecture choice + justification
    print("  Architecture: ???")
    print("  Reason: ???\n")